# Занятие 7. Словари, множества, хеш-таблицы

**План занятия**

1. Словарь и подсчёт
2. Множество и операции над множествами
3. Устройство хеш-таблицы
4. Коллизии
5. Сортировка подсчётом
6. Выбор структуры
7. Домашние задачи

**Теория:** `theory/07_Словари_и_множества.md`
Т. Гэддис, гл. 9 (с. 472 / PDF 497)
А. Бхаргава, гл. 5 (с. 100 / PDF 101)

In [ ]:
# Эта ячейка находит папку с данными. Запустите её ПЕРВОЙ.
import os

CANDIDATES = ["../data", "data", "./data", "/content/data",
              "/content/drive/MyDrive/mglu/data"]
DATA = next((p for p in CANDIDATES if os.path.isdir(p)), None)
print("Data folder:", os.path.abspath(DATA) if DATA else "NOT FOUND")

---

## 1. Словарь

In [ ]:
counts = {"a": 42, "b": 17}

print(counts["a"])
counts["c"] = 5
counts["a"] += 1
print(counts, len(counts))
print("a" in counts)

try:
    print(counts["missing"])
except KeyError as error:
    print("KeyError:", error)

In [ ]:
print(counts.get("missing"))          # None
print(counts.get("missing", 0))      # значение по умолчанию

Подсчёт частот, три способа.

In [ ]:
import string

PUNCT = string.punctuation + "\u00ab\u00bb\u2014\u2013\u2026"

with open(f"{DATA}/text_clean.txt", encoding="utf-8") as f:
    words = [w.strip(PUNCT).lower() for w in f.read().split()]
words = [w for w in words if w]

print("слов:", len(words))

# способ 1: get с умолчанием
counts = {}
for word in words:
    counts[word] = counts.get(word, 0) + 1

# способ 2: Counter
from collections import Counter
counter = Counter(words)

# способ 3: defaultdict
from collections import defaultdict
dd = defaultdict(int)
for word in words:
    dd[word] += 1

print("совпадают:", counts == dict(counter) == dict(dd))
print("уникальных:", len(counts))

In [ ]:
for word, n in counter.most_common(10):
    print(f"{word:>15} {n:>4}")

Сортировка словаря по значению без `Counter`:

In [ ]:
top = sorted(counts.items(), key=lambda pair: pair[1], reverse=True)[:5]
print(top)

---

## 2. Множество

In [ ]:
unique = set(words)
print("уникальных:", len(unique))

print(type({}), type(set()))     # {} это пустой СЛОВАРЬ

unique.add("test")
unique.discard("test")
unique.discard("no such word")   # без ошибки
print("test" in unique)

In [ ]:
with open(f"{DATA}/stopwords_ru.txt", encoding="utf-8") as f:
    stopwords = {w.strip() for w in f if w.strip()}

vocabulary = set(words)

print("словарь текста:", len(vocabulary))
print("стоп-слов:", len(stopwords))
print()
print("пересечение:", len(vocabulary & stopwords))
print("знаменательная лексика:", len(vocabulary - stopwords))
print("объединение:", len(vocabulary | stopwords))

---

## 3. Хеш-таблица

Словарь вычисляет из ключа число и по нему знает, в какую ячейку смотреть.

In [ ]:
for key in ["text", "text", 42, (1, 2), True]:
    print(f"{str(key):>8} -> {hash(key)}")

print()
print("Хеш строк рандомизируется при каждом запуске Python.")
print("Внутри одного запуска он постоянен, этого достаточно.")

In [ ]:
try:
    hash([1, 2])
except TypeError as error:
    print("TypeError:", error)
    print("список изменяем, его хеш устарел бы после изменения")
    print("кортеж неизменяем:", hash((1, 2)))

Ключом может быть только неизменяемое значение. Отсюда практическая ценность
кортежей с занятия 6.

---

## 4. Коллизии

Ячеек конечное число, ключей сколько угодно. Смоделируем таблицу из десяти ячеек.

In [ ]:
def buckets(keys, size=10):
    table = [[] for _ in range(size)]
    for key in keys:
        table[hash(key) % size].append(key)
    return table

sample = sorted(vocabulary)[:12]
for i, bucket in enumerate(buckets(sample)):
    if bucket:
        mark = "  <-- коллизия" if len(bucket) > 1 else ""
        print(f"ячейка {i}: {bucket}{mark}")

При коллизии в ячейке лежит список, и Python дочитывает его перебором.
Отсюда уточнение: в среднем поиск O(1), в худшем случае O(n).

Худший случай на практике не встречается: хеш-функция хорошая, таблица
расширяется при заполнении больше чем на две трети.

In [ ]:
# на настоящем словаре с большим модулем коллизий почти нет
for size in [16, 64, 512, 4096]:
    table = buckets(sorted(vocabulary), size)
    collisions = sum(1 for b in table if len(b) > 1)
    print(f"ячеек {size:>5}: корзин с коллизиями {collisions:>4} "
          f"из занятых {sum(1 for b in table if b):>4}")

---

## 5. Сортировка подсчётом

Счётчик работает и как алгоритм сортировки, когда значения целые
из небольшого известного диапазона.

In [ ]:
def counting_sort(values, low, high):
    counts = [0] * (high - low + 1)
    for value in values:
        counts[value - low] += 1

    result = []
    for i, count in enumerate(counts):
        result.extend([i + low] * count)
    return result

print(counting_sort([3, 1, 5, 1, 3, 3, 2], 1, 5))

In [ ]:
import random, time

random.seed(0)
grades = [random.randint(1, 5) for _ in range(2_000_000)]

start = time.perf_counter()
a = sorted(grades)
builtin_time = time.perf_counter() - start

start = time.perf_counter()
b = counting_sort(grades, 1, 5)
counting_time = time.perf_counter() - start

print(f"sorted(), O(n log n):    {builtin_time:.3f} s")
print(f"подсчётом, O(n + k):     {counting_time:.3f} s")
print(f"ratio: {builtin_time / counting_time:.1f}x")
print("равны:", a == b)

| Данные | Подходит |
|---|---|
| оценки 1..5, возраст 0..120, длины слов | да |
| зарплаты, частоты | нет, диапазон огромен |
| строки, даты, дробные числа | нет |

Массив счётчиков это одновременно и гистограмма распределения.

In [ ]:
lengths = [len(w) for w in words]
distribution = Counter(lengths)

for length in sorted(distribution):
    share = distribution[length] / len(lengths)
    bar = "#" * int(share * 150)
    print(f"{length:>2}: {bar:<40} {distribution[length]:>4} ({share:.1%})")

---

## 6. Выбор структуры

| | `list` | `set` | `dict` |
|---|---|---|---|
| порядок | да | нет | по добавлению |
| повторы | да | нет | ключи уникальны |
| `x in c` | O(n) | O(1) | O(1) по ключу |
| главный вопрос | что третье по счёту | есть ли такое | что связано с этим |

```
нужно связать одно с другим        -> dict
нужно только знать, есть ли        -> set
важен порядок или нужны повторы    -> list
```

---

# Домашние задачи

Рассчитаны примерно на 30 минут.

### Задача 1. Трассировка (без запуска)

Что напечатает код?

In [ ]:
# Мой ответ: ...

# d = {}
# print(d.get("x"))
# print(d.get("x", 0))
# d["x"] = d.get("x", 0) + 1
# print(d)
# print(len({"a", "b", "a", "b"}))
# print(type({}), type(set()))

### Задача 2. Минимум LeetCode

**LeetCode 1 Two Sum** плюс письменный разбор.

Сначала напишите наивное решение двойным циклом, оно очевидное и правильное.
Потом спросите себя, что делается на каждом шаге внутреннего цикла.
Ответ: «ищу, встречалось ли мне нужное число». Это словарь.
В разборе опишите именно этот переход.

In [ ]:
def two_sum_slow(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return []

def two_sum(nums, target):
    # ваш код здесь, через словарь
    pass

# print(two_sum_slow([2, 7, 11, 15], 9))
# print(two_sum([2, 7, 11, 15], 9))

### Задача 3. Сравнение двух документов

Возьмите два файла из `data/corpus/`. Посчитайте общую лексику, лексику,
уникальную для каждого, и отношение размера пересечения к размеру объединения.

In [ ]:
import glob

paths = sorted(glob.glob(f"{DATA}/corpus/*.txt"))
print(paths[:3])

# ваш код здесь

### Задача 4. Распределение длин предложений

Постройте распределение длин предложений в словах для `text_clean.txt`.
Границей предложения считайте точку, восклицательный и вопросительный знаки.
Подходит ли здесь сортировка подсчётом и почему.

In [ ]:
# ваш код здесь

### Задача 5. Подумать (кода не нужно)

Вы считаете частоты по корпусу в 10 миллионов слов. Словарь занимает много памяти.
Какие три способа сократить расход вы предложите и что каждый из них теряет?

### Задача 6. Трек «алгоритмы» (по желанию)

242 Valid Anagram, 387 First Unique Character in a String,
1207 Unique Number of Occurrences, 1636 Sort Array by Increasing Frequency,
349 Intersection of Two Arrays.

---

# Итоги

- `{}` создаёт пустой словарь, множество создаётся через `set()`.
- `get` с умолчанием избавляет от `KeyError`.
- `Counter` и `defaultdict` делают то же, что цикл с `get`.
- Ключом может быть только неизменяемое значение.
- O(1) у словаря это обещание в среднем, коллизии существуют.
- Сортировка подсчётом требует целых значений из небольшого диапазона.
- Массив счётчиков это гистограмма.